## Linking in Spark


<a target="_blank" href="https://colab.research.google.com/github/RobinL/splink/blob/ipynbs/docs/demos/examples/spark/deduplicate_1k_synthetic.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


In [1]:
# Uncomment and run this cell if you're running in Google Colab.
# !pip install "splink[altair,igraph,pyarrow] @ git+https://github.com/RobinL/splink.git@master"
# !pip install pyspark

In [2]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession

from splink.backends.spark import similarity_jar_location

conf = SparkConf()
# This parallelism setting is only suitable for a small toy example
conf.set("spark.driver.memory", "12g")
conf.set("spark.default.parallelism", "8")
conf.set("spark.sql.codegen.wholeStage", "false")


# Add custom similarity functions, which are bundled with Splink
# documented here: https://github.com/moj-analytical-services/splink_scalaudfs
path = similarity_jar_location()
conf.set("spark.jars", path)

sc = SparkContext.getOrCreate(conf=conf)

spark = SparkSession(sc)
spark.sparkContext.setCheckpointDir("./tmp_checkpoints")

26/09/18 10:51:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/runner/work/splink/splink/docs/demos/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
# Disable warnings for pyspark - you don't need to include this
import warnings

spark.sparkContext.setLogLevel("ERROR")
warnings.simplefilter("ignore", UserWarning)

In [4]:
from splink import splink_datasets

records = splink_datasets.fake_1000.to_pylist()

df = spark.createDataFrame(records)

In [5]:
import splink.comparison_library as cl
from splink import Linker, SettingsCreator, SparkAPI, block_on

settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        cl.NameComparison("first_name"),
        cl.NameComparison("surname"),
        cl.LevenshteinAtThresholds(
            "dob"
        ),
        cl.ExactMatch("city").configure(term_frequency_adjustments=True),
        cl.EmailComparison("email"),
    ],
    blocking_rules_to_generate_predictions=[
        block_on("first_name"),
        "l.surname = r.surname",  # alternatively, you can write BRs in their SQL form
    ],
    retain_intermediate_calculation_columns=True,
    em_convergence=0.01,
)

In [6]:
db_api = SparkAPI(spark_session=spark)
df_sdf = db_api.register(df)
linker = Linker(df_sdf, settings)
deterministic_rules = [
    "l.first_name = r.first_name and levenshtein(r.dob, l.dob) <= 1",
    "l.surname = r.surname and levenshtein(r.dob, l.dob) <= 1",
    "l.first_name = r.first_name and levenshtein(r.surname, l.surname) <= 2",
    "l.email = r.email",
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.6)

Probability two random records match is estimated to be  0.00389.
This means that amongst all possible pairwise record comparisons, one in 257.25 are expected to match.  With 499,500 total possible comparisons, we expect a total of around 1,941.67 matching pairs


In [7]:
linker.training.estimate_u_using_random_sampling(max_pairs=5e5)

----- Estimating u probabilities using random sampling -----


Estimating u with: max_pairs = 500,000, min_count_per_level = 100, num_chunks = 10



Estimating u for: first_name (Comparison 1 of 5)


  Running probe chunk (~1.00% of max_pairs)


  Min u_count: 2 for comparison level Jaro-Winkler distance of first_name >= 0.88 (cvv=2)


...

  Running chunk 8/10


  Count of 117 for level Jaro-Winkler >0.88 on username (cvv=1). Chunk took 0.6 seconds.


  Exiting early since min count of 117 exceeds min_count_per_level = 100



Estimated u probabilities using random sampling



Your model is not yet fully trained. Missing estimates for:
    - first_name (no m values are trained).
    - surname (no m values are trained).
    - dob (no m values are trained).
    - city (no m values are trained).
    - email (no m values are trained).


In [8]:
training_blocking_rule = "l.first_name = r.first_name and l.surname = r.surname"
training_session_fname_sname = (
    linker.training.estimate_parameters_using_expectation_maximisation(training_blocking_rule)
)

training_blocking_rule = "l.dob = r.dob"
training_session_dob = linker.training.estimate_parameters_using_expectation_maximisation(
    training_blocking_rule
)


----- Starting EM training session -----



[EM sampling] max_pairs is None — no sampling will be applied


Estimating the m probabilities of the model by blocking on:
l.first_name = r.first_name and l.surname = r.surname

Parameter estimates will be made for the following comparison(s):
    - dob
    - city
    - email

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - first_name
    - surname


Level Jaro-Winkler >0.88 on username on comparison email not observed in dataset, unable to train m value



Iteration 1: Largest change in params was -0.539 in the m_probability of dob, level `Exact match on dob`


...

Iteration 4: Largest change in params was 0.0172 in probability_two_random_records_match


Iteration 5: Largest change in params was 0.00707 in probability_two_random_records_match



EM converged after 5 iterations


m probability not trained for email - Jaro-Winkler >0.88 on username (comparison vector value: 1). This usually means the comparison level was never observed in the training data.



Your model is not yet fully trained. Missing estimates for:
    - email (some m values are not trained).


In [9]:
results = linker.inference.predict(threshold_match_probability=0.9)

Blocking time: 0.69 seconds


Predict time (post-blocking): 2.27 seconds



 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained parameters will use default values.
Comparison: 'email':
    m values not fully trained


In [10]:
spark_df = results.as_spark_dataframe().show()

+------------------+------------------+-----------+-----------+------------+------------+----------------+--------------------+--------------------+-----------------+--------------------+---------+----------+-------------+--------------------+--------------------+------------------+--------------------+----------+----------+---------+-------------------+-------------------+-------------------+----------+--------------------+--------------------+-------------------+------------------+--------------------+--------------------+-----------+--------------------+--------------------+------------------+-------------------+---------+
|      match_weight| match_probability|unique_id_l|unique_id_r|first_name_l|first_name_r|gamma_first_name|     tf_first_name_l|     tf_first_name_r|    mw_first_name|mw_tf_adj_first_name|surname_l| surname_r|gamma_surname|        tf_surname_l|        tf_surname_r|        mw_surname|   mw_tf_adj_surname|     dob_l|     dob_r|gamma_dob|             mw_dob|          